The following project is based on a classification problem which involves loan approval prediction. The dataset used in the project was extracted from Kaggle: https://www.kaggle.com/datasets/udaymalviya/bank-loan-data/data.

However for every step and code in the project that is not referenced was done independently and I take full responsibility of ownership. 

The dataset contains about 45000 obverations of loan applicants and various features or attributes which includes; age, gender, level of education, loan amount and so on with that target attribute as loan status which shows 1 if the loan was approved and 0 if not approved.

This we will explore in our Exploratory Data Analysis Section. We also to preprocess our dataset were necesssary to get it ready for modelling.

In the project we have apply two classiffication algorithms using python.

In [1]:
#Importing Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
#Loading dataset
df = pd.read_csv('loan_data.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'loan_data.csv'

#### Exploratory Data Analysis

In [ ]:
# Viewing the dataset
df.head()

In [ ]:
#Summary of the dataset
df.info()

In [ ]:
#Data Shape
df.shape

In [ ]:
# statistical summary of the numerical columns
df.describe(include = 'all')

In [ ]:
# Checking for missing values
df.isna().sum()

In [ ]:
#Checking for duplicates
df.duplicated().sum()

In [ ]:
# Viewing the distribution of Age
sns.histplot(data=df, x='person_age', bins=10)
plt.title('Age Distribution')
plt.xlabel('Age')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Viewing the distribution of Age and the loan_status
sns.histplot(data=df, x='person_age', hue='loan_status', bins=10)
plt.title('Age Distribution')
plt.xlabel('Age')
plt.ylabel('Frequency')
plt.show()

In [ ]:
#Viewing the gender counts
sns.countplot(data=df, x='person_gender')
plt.xlabel('Gender')
plt.ylabel('Frequency')
plt.show()

print(df['person_gender'].value_counts())

In [ ]:
sns.countplot(data=df, x='person_gender', hue = 'loan_status')
plt.xlabel('Gender')
plt.ylabel('Frequency')
plt.show()

In [ ]:
#Viewing the intent for loan
df['loan_intent'].value_counts().sort_values()

In [ ]:
# Viewing the Count of the intent and loan status
plt.figure(figsize=(12, 4))
sns.countplot(data=df, x='loan_intent', hue ='loan_status')
plt.xlabel('Loan Intentions')
plt.ylabel('Frequency')
plt.show()

The percentage gives us a better understanding, showing the rate of approval based on the intent, with 'Debt Consolidations' having the most approval rate of 30%, followed by medical purposes with ~28% with the rest making up the number.

In [ ]:
pct_approved = df[df['loan_status'] == 1].groupby('loan_intent').size()/(df.groupby('loan_intent').size())*100
pct_approved = pct_approved.round(1)
#.astype(str) + '%'
pct_approved.sort_values(ascending=False)

In [ ]:
# previous loan default percentage table
default_per = pd.crosstab(df['previous_loan_defaults_on_file'], df['loan_status'], normalize='index') * 100
default_per

In [ ]:
# plotting in a bar chart
default_per.plot(kind='bar', stacked=True)
plt.title('Percentage of Loan Status by Previous Loan Default')
plt.xlabel('Previous Loan Default')
plt.ylabel('Percentage (%)')
plt.show()
print(default_per.round(2))

Using the stacked bar plot while comparing the relationship of 'previous loan default on file' feature with our target variable. As we clearly observed that previous loan default have no chance for loan approval which is logically correct, this explain better when people default on previous loans they tends not to be eligible for approval. This feature play a significant role in our target variable (loan_status) which also shows that people with clean record or past defaults are more likely to get approval.

In [ ]:
# Plotting the Correlation Matrix

num_var = df.select_dtypes(include = ['float64', 'int'])
corr_mat = num_var.corr()

low_tri = np.triu(np.ones_like(corr_mat, dtype=bool), k=1)
plt.figure(figsize=(15, 8))
sns.heatmap(data=corr_mat, annot=True, square=False, mask=low_tri)
plt.show()


Interpretation of the Correlation Matrix
This correlation matrix displays the strength and direction of relationships between the numerical variables in the dataset. The values range from -1 to +1:

+1 indicates a perfect positive relationship

-1 indicates a perfect negative relationship

0 means no linear relationship

In [ ]:
plt.figure(figsize=(8,6))
sns.boxplot(data=df, x='loan_status', y='credit_score', palette='Set1')
plt.title('Credit Score Distribution by Loan Status')
plt.xlabel('Loan Status')
plt.ylabel('Credit Score')
plt.show()

In [ ]:
df['loan_status'].value_counts()

From the visualization of the boxplot of our loan_status we can see that this looks more confusing and doesn't seem to have a significant explanation of the relationship of the credit_score to the loan staus. This can be as a result of having an imbalance dataset which may affect the distribution. The value_counts of our loan_status confirms the imbalance between our approved and not approved. For a this project we use the SMOTE method which is one of the reliable method of balancing data. The SMOTE avoid duplicating the data, it generates synthetic samples for the minority class while also avoiding overfitting.

### Feature Selection and Modelling

For the application of the SMOTE method; this requires all features in a dataset to be numerical. In the Preparation of our dataset for training, knowing that Machine Learning models understand only numeric values, we must convert the categorical values into numeric format. Our data contains some columns with categorical data like the person_gender and previous_loan_defaults_on_file which we have easily replaced with values of 0 and 1, because the have only 2 categorically values or boolean values. However, in the case of the person_education, personal_home_ownership and loan_intent, these are categorical data with more than 3 values in the columns with no ranks or order. To achieve this we introduce the One-Hot Encoding which is a method for converting categorical variables into a numerical format without introducing unintended relationships.

##### What is one-hot encoding?

One-hot encoding is a data encoding technique used to convert categorical features of a dataset into numeric features. If a categorical feature has N unique values, we create N new binary columns in the dataset for one-hot encoding. Each new column represents a unique value in the existing categorical feature. If the existing categorical column contains the value represented by a new column, the value in the new column is set to 1. Values in the rest of the new columns are set to 0. [1]

In [ ]:
# Replacing the categorical value of gender and previous loan default to 0 and 1
df['person_gender'] = df['person_gender'].replace(['male', 'female'], [0,1])
df['previous_loan_defaults_on_file'] = df['previous_loan_defaults_on_file'].replace(['No', 'Yes'], [0,1])

In [ ]:
# Converting to numerical using OneHotEncoder 
df = pd.get_dummies(df, columns=['person_home_ownership', 'loan_intent'], dtype=int, drop_first=True)

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

ordinal_cat = ['High School', 'Associate', 'Bachelor', 'Master', 'Doctorate']
# Create the encoder
enc = OrdinalEncoder(categories=[ordinal_cat])

# Applying it
df[['person_education']] = enc.fit_transform(df[['person_education']])

In [ ]:
#Viewing the dtypes
df.info()

In [ ]:
# Importing SMOTE
from imblearn.over_sampling import SMOTE


# Splitting and Balancing the dataset
X = df.drop('loan_status', axis=1)
y = df['loan_status']

sm = SMOTE(random_state=42)
X_bal, y_bal = sm.fit_resample(X, y)

#Concatenating the dataset 
df = pd.concat([X_bal, y_bal], axis=1)

df['loan_status'].value_counts()

As stated this project is based on a classification problem where the objective is to classify and to predict if a loan application would be approved or not. We have simply explored our dataset and preprocess it for training, will employ scikit-learn library for classification. We will apply two different classification algorithms, (i.e) KNN and the Logistic regression.

firstly, determining the target Variable and the independent variable and spliting our data into X - independent and y - dependent features


In [ ]:
# Splitting the data
X = df.drop('loan_status', axis=1)
y = df['loan_status']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

In [ ]:
from sklearn.preprocessing import StandardScaler

# Select non binary columns
cont_cols = ['person_age', 'person_income', 'person_emp_exp', 'loan_amnt', 'credit_score', 
             'loan_percent_income', 'loan_int_rate', 'cb_person_cred_hist_length']

# Copying the data
sc = StandardScaler()
X_train_sc = X_train.copy()
X_test_sc = X_test.copy()

# Scaling and Transforming 
X_train_sc[cont_cols] = sc.fit_transform(X_train[cont_cols])
X_test_sc[cont_cols] = sc.transform(X_test[cont_cols])

#### Training

#### K-Nearest Neighbor

In [ ]:
#Importing KNN and applying model on our training set
from sklearn.neighbors import KNeighborsClassifier
classifier = KNeighborsClassifier(n_neighbors=5, metric='minkowski', p=2)
classifier.fit(X_train_sc, y_train)

In [ ]:
y_pred = classifier.predict(X_test_sc)
print(y_pred)

In [ ]:
#Evaluating Prediction
from sklearn import metrics
acc = metrics.accuracy_score(y_test,y_pred)
print('accuracy:', round(acc, 2))
cm = metrics.confusion_matrix(y_test,y_pred)
print('confusion matrix:')
print(cm,'\n\n')
result = metrics.classification_report(y_test,y_pred)
print('Classification Report:\n')
print(result)

In [ ]:
ax = sns.heatmap(cm, cmap='flare', annot=True, fmt='d')
plt.xlabel('Predicted Class', fontsize=10)
plt.ylabel('True Class', fontsize=10)
plt.title(' KNN Confusion Matrix', fontsize=12)
plt.show()

#### Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
#Training the model
logr = LogisticRegression()
logr.fit(X_train_sc, y_train)

#Predict on the test data
y_pred = logr.predict(X_test_sc)

In [ ]:
#Evaluate accuracy
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc*100:.2f}%")
cfm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(cm)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
ax = sns.heatmap(cfm, cmap='flare', annot=True, fmt='d')
plt.xlabel('Predicted Class', fontsize=10)
plt.ylabel('True Class', fontsize=10)
plt.title('Logistic_Regression Confusion Matrix', fontsize=12)
plt.show()